In [ ]:
def compute_hours_since_known_rain(ppt_series):
    """
    Computes HoursSinceRain with a hybrid strategy.
    Dynamically adjusts max_tolerated_nans based on ppt NaN density.
    """

    # Step 1: Compute NaN density
    nan_ratio = ppt_series.isna().mean()

    # Step 2: Decide tolerance level
    if nan_ratio < 0.01:
        max_tolerated_nans = 0
    elif nan_ratio < 0.05:
        max_tolerated_nans = 1
    elif nan_ratio < 0.15:
        max_tolerated_nans = 2
    elif nan_ratio < 0.25:
        max_tolerated_nans = 3
    else:
        max_tolerated_nans = 5

    # Step 3: Apply hybrid logic
    hours_since = []
    count = None
    unknown_streak = 0

    for v in ppt_series:
        if pd.isna(v):
            unknown_streak += 1
            if count is None or unknown_streak > max_tolerated_nans:
                count = None  # We've lost confidence in last known rain
            hours_since.append(np.nan)
        elif v > 0:
            count = 0
            unknown_streak = 0
            hours_since.append(0)
        elif v == 0:
            unknown_streak = 0
            if count is None:
                hours_since.append(np.nan)
            else:
                count += 1
                hours_since.append(count)

    return hours_since


In [ ]:
DEPENDENT_FEATURES = {
    "Ppt": ["Ppt_3h_sum", "Ppt_24h_sum", "Ppt_RainFlag", "HoursSinceRain"]
}
def apply_dependent_masking(mask, feature_list, dependent_features):
    """
    Enforces masking dependencies:
    If a base feature is masked at any timestep, its dependent features are also masked.
    """
    for base_feat, dependents in dependent_features.items():
        if base_feat not in feature_list:
            continue
        base_idx = feature_list.index(base_feat)
        base_mask = mask[:, :, base_idx]

        for dep_feat in dependents:
            if dep_feat in feature_list:
                dep_idx = feature_list.index(dep_feat)
                mask[:, :, dep_idx] |= base_mask

    return mask

mask_type_probs = {
    'random': 0.25,
    'block': 0.25,
    'row': 0.25,
    'col': 0.25
}

def create_per_window_mask(window, tot_mask_amt, mask_type_probs):
    """
    Creates a composite mask for a single window using a randomized distribution
    of mask types based on mask_type_probs.
    """
    alpha = np.array([mask_type_probs[k] for k in ['random', 'block', 'row', 'col']])
    mask_amts = np.random.dirichlet(alpha) * tot_mask_amt

    rand_mask = create_random_mask(window, mask_amts[0])
    block_mask = create_block_mask(window, mask_amts[1])
    row_mask = create_row_mask(window, mask_amts[2])
    col_mask = create_col_mask(window, mask_amts[3])

    combined_mask = rand_mask | block_mask | row_mask | col_mask
    return combined_mask